In [43]:
import pandas as pd
import sys
from qiskit import QuantumCircuit

sys.path.append(
    "/Users/andrewweiland/UCCS_REU/quantum-circuit-chaining"
)
from benchmarks.ai_transpile.transpilers import analyze_circuit
df = pd.read_csv("master.csv", index_col=0)

In [44]:
df

,parent_circuit_id,chain_name,original_circuit,original_circuit_path,opt_1,opt_2,opt_3,qasm_path,initial_num_qubits,initial_depth,initial_two_qubit_gates,initial_total_gates,final_num_qubits,final_depth,final_two_qubit_gates,final_total_gates,opt_chain
id,,,,,,,,,,,,,,,,,
1,1,single_1,efficient_su2_10_r2,benchmarks/ai_transpile/qasm/efficient_su2_10_...,wisq_rules,NaN,NaN,NaN,-1,17,18,78,0,23,18,138,efficient_su2_10_r2__wisq_rules
2,1,single_1,efficient_su2_10_r2,benchmarks/ai_transpile/qasm/efficient_su2_10_...,wisq_rules,NaN,NaN,NaN,-1,17,18,78,0,23,18,138,efficient_su2_10_r2__wisq_rules
3,1,single_2,efficient_su2_10_r2,benchmarks/ai_transpile/qasm/efficient_su2_10_...,wisq_bqskit,NaN,NaN,NaN,-1,17,18,78,0,45,13,73,efficient_su2_10_r2__wisq_bqskit
4,1,single_2,efficient_su2_10_r2,benchmarks/ai_transpile/qasm/efficient_su2_10_...,wisq_bqskit,NaN,NaN,NaN,-1,17,18,78,0,34,12,59,efficient_su2_10_r2__wisq_bqskit
5,1,single_2,efficient_su2_10_r2,benchmarks/ai_transpile/qasm/efficient_su2_10_...,wisq_bqskit,NaN,NaN,NaN,-1,17,18,78,0,27,12,45,efficient_su2_10_r2__wisq_bqskit
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16555,1266,chain3_wisq_rules__wisq_bqskit__wisq_bqskit,mod5_4,benchmarks/ai_transpile/qasm/feynman/mod5_4.qasm,wisq_rules,wisq_bqskit,wisq_bqskit,NaN,-1,23,8,23,0,71,18,127,mod5_4__wisq_rules__wisq_bqskit__wisq_bqskit
16556,1266,chain3_wisq_rules__wisq_bqskit__wisq_rules,mod5_4,benchmarks/ai_transpile/qasm/feynman/mod5_4.qasm,wisq_rules,wisq_bqskit,wisq_rules,NaN,-1,23,8,23,0,83,18,133,mod5_4__wisq_rules__wisq_bqskit__wisq_rules
16557,1267,chain3_wisq_rules__wisq_rules__qiskit_ai,mod5_4,benchmarks/ai_transpile/qasm/feynman/mod5_4.qasm,wisq_rules,wisq_rules,qiskit_ai,NaN,-1,23,8,23,0,77,37,94,mod5_4__wisq_rules__wisq_rules__qiskit_ai


In [45]:
og_circuits = df["original_circuit"].dropna().unique().tolist()
len(og_circuits)

106

In [46]:
def is_exhaustive(df, optimizers):
    optimizers_tested = set(df["opt_1"].dropna().unique().tolist())
    if optimizers == optimizers_tested:
        return -1
    return list(set(optimizers) - set(optimizers_tested))

    

In [48]:
optimizers = set(["wisq_rules", "wisq_bqskit", "tket", "qiskit_ai", "qiskit_standard"])

not_exhaustive = {}
for circuit in og_circuits:
    temp_df = df[
        (df["original_circuit"] == circuit)
        & (df["opt_2"].isna())
        & (df["opt_3"].isna())
        ]    
    exhaustive = is_exhaustive(temp_df, optimizers)
    if exhaustive != -1:
        print(f"{circuit} has not been tested exhaustively.")
        print(f"Untested optimizers: {exhaustive}\n")
        not_exhaustive[circuit] = exhaustive 

        
    path = "../quantum-circuit-chaining/" + temp_df.iloc[0]["original_circuit_path"]
    original_qc = QuantumCircuit.from_qasm_file(path)
    metrics = analyze_circuit(original_qc)
    temp_df = temp_df.sort_values(
            by=['final_two_qubit_gates', 'final_depth']
    )
    print(f"Original Metrics for: {circuit}")
    print(metrics)
    print(f"Tested Optimizers ranked for {circuit}")
    for _, row in temp_df.iterrows():
        print(f"{row['opt_chain']}:")
        print(f"     2Q Circuits: {row['final_two_qubit_gates']}")
        print(f"     Depth:       {row['final_depth']}")
        break

    print("\n\n\n")
        

Original Metrics for: efficient_su2_10_r2
CircuitMetrics(depth=17, two_qubit_gates=18, two_qubit_depth=11, total_gates=78)
Tested Optimizers ranked for efficient_su2_10_r2
efficient_su2_10_r2__wisq_bqskit:
     2Q Circuits: 12
     Depth:       27




Original Metrics for: efficient_su2_12
CircuitMetrics(depth=25, two_qubit_gates=66, two_qubit_depth=21, total_gates=114)
Tested Optimizers ranked for efficient_su2_12
efficient_su2_12__wisq_bqskit:
     2Q Circuits: 11
     Depth:       27




Original Metrics for: efficient_su2_12_r2
CircuitMetrics(depth=19, two_qubit_gates=22, two_qubit_depth=13, total_gates=94)
Tested Optimizers ranked for efficient_su2_12_r2
efficient_su2_12_r2__wisq_bqskit:
     2Q Circuits: 16
     Depth:       39




Original Metrics for: efficient_su2_16
CircuitMetrics(depth=33, two_qubit_gates=120, two_qubit_depth=29, total_gates=184)
Tested Optimizers ranked for efficient_su2_16
efficient_su2_16__wisq_bqskit:
     2Q Circuits: 15
     Depth:       47




Origina

In [49]:
print(not_exhaustive)

{'W-state': ['wisq_rules', 'wisq_bqskit'], 'adder': ['wisq_rules', 'wisq_bqskit'], 'bigadder': ['wisq_rules', 'wisq_bqskit'], 'ham15-med': ['qiskit_ai'], 'hwb10': ['qiskit_ai'], 'hwb11': ['qiskit_ai'], 'hwb12': ['qiskit_ai'], 'hwb8': ['qiskit_ai'], 'qpt': ['wisq_rules', 'wisq_bqskit'], 'qft_16': ['qiskit_ai'], 'qv_N015_12345': ['qiskit_ai'], 'qv_N017_12345': ['qiskit_ai'], 'qv_N018_12345': ['qiskit_ai'], 'qv_N019_12345': ['qiskit_ai']}
